## 일관성 있는 파일 만들기
### overview : 모든 데이터셋에서 있는 파일만 살리기

mask있는 폴더들 내에서 image와 mask 개수 통일 -> 생각해보니 할 필요없잖아..?



1. 모든 데이터들 다 불러오기
    - mask있는 폴더들은 mask따로 image따로
    - train, test, val은 합치기
1. 모든 데이터 list
    - 오염 origin 이미지
    - 오염 seg 이미지
    - 구름 origin 이미지
    - 구름 seg 이미지
    - table
    
    
table과 모든 것을 비교해서 다 없애고, table을 기준으로 나머지를 없앤다

### dataset folder 설정

In [1]:
dataset_folders = {
    "soiling" : {
        "seg":"/app/pytorch-fcn/examples/voc/logs/20230831_174351.813277/result1/",
        "origin" : "./dataset/image/20230605_ch2_crop_resize_212/"
    },
    "cloud" :{
        "seg" : "./segmentationModel/results/20230605_ch1_crop_mask/FCN/",
        "origin" : "./dataset/image/20230605_ch1_crop/test/Image/",
        "classification" : "./cloud_classification/"
    },
    "table" : "./dataset/table/",
}

### table 한 줄로 펴기

In [2]:
import pandas as pd 

# 모든 행 출력
pd.set_option('display.max_rows', None)

df_table = pd.read_csv(f'{dataset_folders["table"]}weather-addSoilingType_examine_1_upload.csv')
df_table['일시']=pd.to_datetime(df_table['일시'])
df_table

,일시,기온(°C),풍속(m/s),습도(%),누적일사(MJ/m^2),일사(MJ/m^2),패널1,패널2,패널3,패널4,패널1감소량(%),패널2감소량(%),패널3감소량(%),panel1_type,panel2_type,panel3_type,비고
0,2023-03-22 16:44:00,24.8,0.6,35.1,19.03,0.01,30.253643,24.392439,43.047188,47.617451,36.465219,48.774160,9.597874,sand,sand,-,NaN
1,2023-03-22 16:48:00,24.8,0.3,35.5,19.11,0.02,18.711142,28.045124,41.918766,45.050343,58.466150,37.747146,6.951282,sand,sand,-,NaN
2,2023-03-22 16:52:00,25.1,1.6,35.4,19.18,0.03,18.002813,17.785406,43.793674,50.285156,64.198553,64.630902,12.909340,sand,sand,-,NaN
3,2023-03-22 16:56:00,24.7,1.9,35.6,19.23,0.02,14.933952,4.243219,45.630005,47.265797,68.404315,91.022644,3.460836,sand,sand,-,NaN
4,2023-03-22 17:02:00,24.6,2.2,36.0,19.29,0.00,23.847889,0.904453,41.216297,42.699940,44.150064,97.881840,3.474578,sand,sand,-,NaN
5,2023-03-29 09:17:00,8.8,0.4,55.2,2.17,0.03,79.253433,174.313141,159.924744,234.059204,66.139578,25.526047,31.673380,shadow,pine_pollen,pine_pollen,NaN
6,2023-03-29 09:21:00,8.9,0.4,56.2,2.28,0.03,225.318741,184.460068,117.164162,239.669708,5.987810,23.035719,51.114322,-,pine_pollen,pine_pollen,NaN
7,2023-03-29 09:25:00,9.2,0.2,55.4,2.42,0.03,96.024139,174.675369,122.919571,228.208313,57.922594,23.457929,46.137120,shadow,pine_pollen,pine_pollen,NaN
8,2023-03-29 09:29:00,9.4,0.7,55.9,2.53,0.03,74.881966,145.722946,107.325569,204.925507,63.458934,28.889796,47.627032,shadow,pine_pollen,pine_pollen,NaN
9,2023-03-29 09:33:00,9.4,0.1,54.3,2.65,0.04,192.788422,135.386063,82.275703,204.972046,5.944042,33.949011,59.860037,-,pine_pollen,pine_pollen,NaN


In [3]:
df_panels = []


weather_cols = ['일시', '기온(°C)', '풍속(m/s)', '습도(%)', '일사(MJ/m^2)']

for i in range(1,4):
    panel_cols= [f'패널{i}', f'패널{i}감소량(%)', f'panel{i}_type']
    df_panels.append(df_table[weather_cols+panel_cols])

    
panel_cols= ['패널4']
df_panels.append(df_table[weather_cols+panel_cols])  

In [4]:
for i in range(3):
    panel_cols= [f'패널{i+1}', f'패널{i+1}감소량(%)', f'panel{i+1}_type']
    df_panels[i].rename(columns = {f'패널{i+1}' : 'power generation',
                                  f'패널{i+1}감소량(%)' : 'reduction amount(%)',
                                  f'panel{i+1}_type' : 'soiling type'}, inplace = True)
    df_panels[i].insert(6,'panel number', [i+1 for _ in range(len(df_panels[i]))])
    
df_panels[3].rename(columns = {f'패널{4}' : 'power generation'},inplace = True)
df_panels[3].insert(6,'panel number',[4 for _ in range(len(df_panels[i]))])
df_panels[3]['soiling type'] = ['clean' for _ in range(len(df_panels[i]))]

/usr/local/lib/python3.8/dist-packages/pandas/core/frame.py:4441: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().rename(
/tmp/ipykernel_43240/719428071.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_panels[3]['soiling type'] = ['clean' for _ in range(len(df_panels[i]))]


In [5]:
df= pd.concat(df_panels)
df

,일시,기온(°C),풍속(m/s),습도(%),일사(MJ/m^2),power generation,panel number,reduction amount(%),soiling type
0,2023-03-22 16:44:00,24.8,0.6,35.1,0.01,30.253643,1,36.465219,sand
1,2023-03-22 16:48:00,24.8,0.3,35.5,0.02,18.711142,1,58.466150,sand
2,2023-03-22 16:52:00,25.1,1.6,35.4,0.03,18.002813,1,64.198553,sand
3,2023-03-22 16:56:00,24.7,1.9,35.6,0.02,14.933952,1,68.404315,sand
4,2023-03-22 17:02:00,24.6,2.2,36.0,0.00,23.847889,1,44.150064,sand
5,2023-03-29 09:17:00,8.8,0.4,55.2,0.03,79.253433,1,66.139578,shadow
6,2023-03-29 09:21:00,8.9,0.4,56.2,0.03,225.318741,1,5.987810,-
7,2023-03-29 09:25:00,9.2,0.2,55.4,0.03,96.024139,1,57.922594,shadow
8,2023-03-29 09:29:00,9.4,0.7,55.9,0.03,74.881966,1,63.458934,shadow
9,2023-03-29 09:33:00,9.4,0.1,54.3,0.04,192.788422,1,5.944042,-


In [6]:
df.keys()

Index(['일시', '기온(°C)', '풍속(m/s)', '습도(%)', '일사(MJ/m^2)', 'power generation',
       'panel number', 'reduction amount(%)', 'soiling type'],
      dtype='object')

In [7]:
df = df.reset_index(drop =True)
df = df.dropna(subset=['일시', '기온(°C)', '풍속(m/s)', '습도(%)', '일사(MJ/m^2)', 'power generation',
       'panel number'], axis=0)
df

,일시,기온(°C),풍속(m/s),습도(%),일사(MJ/m^2),power generation,panel number,reduction amount(%),soiling type
0,2023-03-22 16:44:00,24.8,0.6,35.1,0.01,30.253643,1,36.465219,sand
1,2023-03-22 16:48:00,24.8,0.3,35.5,0.02,18.711142,1,58.466150,sand
2,2023-03-22 16:52:00,25.1,1.6,35.4,0.03,18.002813,1,64.198553,sand
3,2023-03-22 16:56:00,24.7,1.9,35.6,0.02,14.933952,1,68.404315,sand
4,2023-03-22 17:02:00,24.6,2.2,36.0,0.00,23.847889,1,44.150064,sand
5,2023-03-29 09:17:00,8.8,0.4,55.2,0.03,79.253433,1,66.139578,shadow
6,2023-03-29 09:21:00,8.9,0.4,56.2,0.03,225.318741,1,5.987810,-
7,2023-03-29 09:25:00,9.2,0.2,55.4,0.03,96.024139,1,57.922594,shadow
8,2023-03-29 09:29:00,9.4,0.7,55.9,0.03,74.881966,1,63.458934,shadow
9,2023-03-29 09:33:00,9.4,0.1,54.3,0.04,192.788422,1,5.944042,-


### soiling 이미지와 비교해서 없는 줄 지우기
#### seg


 "soiling" : {
        "seg":"/app/pytorch-fcn/examples/voc/logs/20230831_174351.813277/result1/",
        "origin" : "./dataset/image/20230516_ch2_crop_resize/"
    }

In [8]:
import os 

# 이름 확인
os.listdir(dataset_folders["soiling"]["seg"])[0:10]

['panel4_NVR_ch2_jpg_20230322164400.png',
 'panel4_NVR_ch2_jpg_20230322164800.png',
 'panel4_NVR_ch2_jpg_20230322165200.png',
 'panel4_NVR_ch2_jpg_20230322165600.png',
 'panel4_NVR_ch2_jpg_20230322170200.png',
 'panel4_NVR_ch2_jpg_20230329091700.png',
 'panel4_NVR_ch2_jpg_20230329092100.png',
 'panel4_NVR_ch2_jpg_20230329092500.png',
 'panel4_NVR_ch2_jpg_20230329092900.png',
 'panel4_NVR_ch2_jpg_20230329093300.png']

In [9]:
import datetime
import glob

indexs = []

for i in df.index :
    date = df.loc[i]['일시'].strftime('%Y%m%d%H%M%S')
    panel = str(df.loc[i]['panel number']) 

    if glob.glob(dataset_folders["soiling"]["seg"]+f"panel{panel}*{date}*") == [] :
        print(i, panel, date)
        indexs.append(i)
df.drop(index=indexs, inplace=True)

6 1 20230329092100
9 1 20230329093300
10 1 20230329093700
152 1 20230410165500
210 1 20230417133800
211 1 20230417133900
212 1 20230417134000
222 1 20230417135400
223 1 20230417135500
224 1 20230417135600
265 1 20230417172500
270 1 20230417173100
337 1 20230424135000
358 1 20230424141900
673 2 20230403135000
693 2 20230403164700
694 2 20230403165100
695 2 20230403165500
696 2 20230403165900
697 2 20230403170300
698 2 20230403170700
699 2 20230403171200
700 2 20230403171800
701 2 20230403172300
702 2 20230403172700
703 2 20230403173100
779 2 20230410165500
837 2 20230417133800
838 2 20230417133900
839 2 20230417134000
849 2 20230417135400
850 2 20230417135500
851 2 20230417135600
892 2 20230417172500
897 2 20230417173100
898 2 20230417173300
899 2 20230417173400
900 2 20230417173500
1254 3 20230322164400
1255 3 20230322164800
1256 3 20230322165200
1257 3 20230322165600
1258 3 20230322170200
1277 3 20230329145800
1278 3 20230329150700
1279 3 20230329164100
1280 3 20230329164500
1281 3 20

In [10]:
df

,일시,기온(°C),풍속(m/s),습도(%),일사(MJ/m^2),power generation,panel number,reduction amount(%),soiling type
0,2023-03-22 16:44:00,24.8,0.6,35.1,0.01,30.253643,1,36.465219,sand
1,2023-03-22 16:48:00,24.8,0.3,35.5,0.02,18.711142,1,58.466150,sand
2,2023-03-22 16:52:00,25.1,1.6,35.4,0.03,18.002813,1,64.198553,sand
3,2023-03-22 16:56:00,24.7,1.9,35.6,0.02,14.933952,1,68.404315,sand
4,2023-03-22 17:02:00,24.6,2.2,36.0,0.00,23.847889,1,44.150064,sand
5,2023-03-29 09:17:00,8.8,0.4,55.2,0.03,79.253433,1,66.139578,shadow
7,2023-03-29 09:25:00,9.2,0.2,55.4,0.03,96.024139,1,57.922594,shadow
8,2023-03-29 09:29:00,9.4,0.7,55.9,0.03,74.881966,1,63.458934,shadow
11,2023-03-29 09:42:00,10.0,0.4,51.2,0.03,33.043358,1,86.974791,sand
12,2023-03-29 14:14:00,20.1,0.8,16.1,0.05,178.110580,1,20.599553,pine_pollen


#### origin

In [11]:
# 이름 확인
os.listdir(dataset_folders["soiling"]["origin"])[0:10]

['panel1_NVR_ch2_jpg_20230322164400.jpg',
 'panel1_NVR_ch2_jpg_20230322164800.jpg',
 'panel1_NVR_ch2_jpg_20230322165200.jpg',
 'panel1_NVR_ch2_jpg_20230322165600.jpg',
 'panel1_NVR_ch2_jpg_20230322170200.jpg',
 'panel1_NVR_ch2_jpg_20230329091700.jpg',
 'panel1_NVR_ch2_jpg_20230329092100.jpg',
 'panel1_NVR_ch2_jpg_20230329092500.jpg',
 'panel1_NVR_ch2_jpg_20230329092900.jpg',
 'panel1_NVR_ch2_jpg_20230329093300.jpg']

In [12]:
indexs = []

for i in df.index :
    date = df.loc[i]['일시'].strftime('%Y%m%d%H%M%S')
    panel = str(df.loc[i]['panel number']) 
    
    if glob.glob(dataset_folders["soiling"]["origin"]+f"panel{panel}*{date}*") == [] :
        print(i, panel, date)
        indexs.append(i)
        
df.drop(index=indexs, inplace=True)

### cloud 이미지와 비교해서 없는 줄 지우기
#### seg


    "cloud" :{
        "seg" : "./segmentationModel/results/20230605_ch1_crop_mask/FCN/",
        "origin" : "./dataset/image/20230605_ch1_crop/test/Image/"
    

In [13]:
# 이름 확인
os.listdir(dataset_folders["cloud"]["seg"])[0:10]

['panel1_NVR_ch1_jpg_20230322164400.jpg_62.9308.jpg',
 'panel1_NVR_ch1_jpg_20230322164800.jpg_79.6429.jpg',
 'panel1_NVR_ch1_jpg_20230322165200.jpg_95.1866.jpg',
 'panel1_NVR_ch1_jpg_20230322165600.jpg_95.0747.jpg',
 'panel1_NVR_ch1_jpg_20230322170200.jpg_92.5118.jpg',
 'panel1_NVR_ch1_jpg_20230329091700.jpg_13.1237.jpg',
 'panel1_NVR_ch1_jpg_20230329092100.jpg_9.8265.jpg',
 'panel1_NVR_ch1_jpg_20230329092500.jpg_11.9565.jpg',
 'panel1_NVR_ch1_jpg_20230329092900.jpg_7.5293.jpg',
 'panel1_NVR_ch1_jpg_20230329093300.jpg_10.9067.jpg']

In [14]:
indexs = []

for i in df.index :
    date = df.loc[i]['일시'].strftime('%Y%m%d%H%M%S')
    panel = str(df.loc[i]['panel number']) 

    if glob.glob(dataset_folders["cloud"]["seg"]+f"*{date}*") == [] :
        print(i, panel, date)
        indexs.append(i)
        
df.drop(index=indexs, inplace=True)

#### origin

In [15]:
# 이름 확인
os.listdir(dataset_folders["cloud"]["origin"])[0:10]

['panel1_NVR_ch1_jpg_20230322164400.jpg',
 'panel1_NVR_ch1_jpg_20230322164800.jpg',
 'panel1_NVR_ch1_jpg_20230322165200.jpg',
 'panel1_NVR_ch1_jpg_20230322165600.jpg',
 'panel1_NVR_ch1_jpg_20230322170200.jpg',
 'panel1_NVR_ch1_jpg_20230329091700.jpg',
 'panel1_NVR_ch1_jpg_20230329092100.jpg',
 'panel1_NVR_ch1_jpg_20230329092500.jpg',
 'panel1_NVR_ch1_jpg_20230329092900.jpg',
 'panel1_NVR_ch1_jpg_20230329093300.jpg']

In [16]:
indexs = []

for i in df.index :
    date = df.loc[i]['일시'].strftime('%Y%m%d%H%M%S')
    panel = str(df.loc[i]['panel number']) 

    if glob.glob(dataset_folders["cloud"]["origin"]+f"*{date}*") == [] :
        print(i, panel, date)
        indexs.append(i)
        
df.drop(index=indexs, inplace=True)

In [17]:
df = df.reset_index(drop =True)
df

,일시,기온(°C),풍속(m/s),습도(%),일사(MJ/m^2),power generation,panel number,reduction amount(%),soiling type
0,2023-03-22 16:44:00,24.8,0.6,35.1,0.01,30.253643,1,36.465219,sand
1,2023-03-22 16:48:00,24.8,0.3,35.5,0.02,18.711142,1,58.466150,sand
2,2023-03-22 16:52:00,25.1,1.6,35.4,0.03,18.002813,1,64.198553,sand
3,2023-03-22 16:56:00,24.7,1.9,35.6,0.02,14.933952,1,68.404315,sand
4,2023-03-22 17:02:00,24.6,2.2,36.0,0.00,23.847889,1,44.150064,sand
5,2023-03-29 09:17:00,8.8,0.4,55.2,0.03,79.253433,1,66.139578,shadow
6,2023-03-29 09:25:00,9.2,0.2,55.4,0.03,96.024139,1,57.922594,shadow
7,2023-03-29 09:29:00,9.4,0.7,55.9,0.03,74.881966,1,63.458934,shadow
8,2023-03-29 09:42:00,10.0,0.4,51.2,0.03,33.043358,1,86.974791,sand
9,2023-03-29 14:14:00,20.1,0.8,16.1,0.05,178.110580,1,20.599553,pine_pollen



### <div class="mark">new dataset folder 설정</div>

In [18]:
new = "./dataset/preprocessed_20230912/"

new_dataset_folders = {
    "soiling" : {
        "seg":f"{new}soiling/seg/",
        "origin" : f"{new}soiling/origin/"
    },
    "cloud" :{
        "seg" : f"{new}cloud/seg/",
        "origin" : f"{new}cloud/origin/",
        "classification" : f"{new}cloud/classification/"
    },
    "table" : f"{new}table/",
}

os.makedirs(new_dataset_folders["soiling"]["seg"], exist_ok=True)
os.makedirs(new_dataset_folders["soiling"]["origin"], exist_ok=True)
os.makedirs(new_dataset_folders["cloud"]["seg"], exist_ok=True)
os.makedirs(new_dataset_folders["cloud"]["origin"], exist_ok=True)
os.makedirs(new_dataset_folders["cloud"]["classification"], exist_ok=True)
os.makedirs(new_dataset_folders["table"], exist_ok=True)

In [19]:
import yaml

with open(os.path.join(new, 'config.yaml'), 'w') as f:
    yaml.safe_dump(
        {"old_dataset_folders" : dataset_folders,
        "new_dataset_folders" : new_dataset_folders
        }, f, default_flow_style=False)

### df 저장하기

In [20]:
df.to_csv(f'{new_dataset_folders["table"]}weather-addSoilingType_examine_1_upload_preprocessed.csv') 

### df(table)를 기반으로 있는 파일만 카피하기

#### soiling

In [21]:
os.listdir(dataset_folders["soiling"]["seg"])[-1]

'panel3_NVR_ch2_jpg_20230524150800.png'

In [22]:
os.listdir(dataset_folders["soiling"]["seg"])[-1][19:-4]

'20230524150800'

In [23]:
import shutil

for f in os.listdir(dataset_folders["soiling"]["seg"]):
    panel=int(f[5]) 
    date = datetime.datetime.strptime(f[19:-4],'%Y%m%d%H%M%S' ) 
    
    if len(df.loc[(df["일시"]==date) & (df["panel number"]==panel)]) != 0 :
        shutil.copy2(dataset_folders["soiling"]["seg"]+f,new_dataset_folders["soiling"]["seg"]+f ) 
    else :
        print(f)

panel4_NVR_ch2_jpg_20230329172100.png
panel4_NVR_ch2_jpg_20230329172800.png
panel4_NVR_ch2_jpg_20230424143300.png
panel1_NVR_ch2_jpg_20230424143300.png
panel2_NVR_ch2_jpg_20230329172800.png
panel2_NVR_ch2_jpg_20230424143300.png


In [24]:
import pandas as pd 


class_names = [        
        'panel',
        'bird dropping',
        'pine pollen',
        'plastic bag',
        'sand',
        'shadow']

s_df = pd.DataFrame(columns=["일시", "panel number"]+class_names)
s_df

,일시,panel number,panel,bird dropping,pine pollen,plastic bag,sand,shadow


In [25]:
from PIL import Image
from tqdm.notebook import tqdm
import numpy as np

cmap = [
    [0,0,0],
    [128,0,0],
    [0,128,0],
    [128,128,0],
    [0,0,128],
    [128,0,128]
]

for file_name in tqdm(os.listdir(new_dataset_folders["soiling"]["seg"])):

    img = Image.open(f'{new_dataset_folders["soiling"]["seg"]}{file_name}')
    
    panel_number = int(file_name[5])
    date = file_name[19:-4]
    
    
    img = np.array(img)
    
    height, width = len(img), len(img[0])
    total = height*width
    
    class_scores = [0 for _ in range(len(class_names))]
    
    for h in range(height):
        for w in range(width):
            class_num = cmap.index(list(img[h][w]))
            
            class_scores[class_num] += 1
            
            
    row = {"일시":date, "panel number":panel_number}
    for i, score in enumerate(class_scores):
        row[class_names[i]] = score/total*100
            
    s_df= s_df.append(row, ignore_index=True)
s_df['일시']=pd.to_datetime(s_df['일시'])
s_df 

  0%|          | 0/2317 [00:00<?, ?it/s]

,일시,panel number,panel,bird dropping,pine pollen,plastic bag,sand,shadow
0,2023-03-22 16:44:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2023-03-22 16:48:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2023-03-22 16:52:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2023-03-22 16:56:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2023-03-22 17:02:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,2023-03-29 09:17:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,2023-03-29 09:21:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,2023-03-29 09:25:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,2023-03-29 09:29:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,2023-03-29 09:33:00,4,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [26]:
s_df = s_df.reset_index(drop =True)
s_df.to_csv(f'{new_dataset_folders["soiling"]["seg"]}segPercentScore.csv') 

---

In [27]:
os.listdir(dataset_folders["soiling"]["origin"])[-1]

'panel4_NVR_ch2_jpg_20230329172800.jpg'

In [28]:
os.listdir(dataset_folders["soiling"]["origin"])[-1][19:-4] 

'20230329172800'

In [29]:
for f in os.listdir(dataset_folders["soiling"]["origin"]):
    panel=int(f[5]) 
    date = datetime.datetime.strptime(f[19:-4],'%Y%m%d%H%M%S' ) 
    
    if len(df.loc[(df["일시"]==date) & (df["panel number"]==panel)]) != 0 :
        shutil.copy2(dataset_folders["soiling"]["origin"]+f,new_dataset_folders["soiling"]["origin"]+f ) 
    else :
        print(f)

panel1_NVR_ch2_jpg_20230329092100.jpg
panel1_NVR_ch2_jpg_20230329093300.jpg
panel1_NVR_ch2_jpg_20230329093700.jpg
panel1_NVR_ch2_jpg_20230329172100.jpg
panel1_NVR_ch2_jpg_20230329172800.jpg
panel1_NVR_ch2_jpg_20230424135000.jpg
panel1_NVR_ch2_jpg_20230424141900.jpg
panel1_NVR_ch2_jpg_20230424143300.jpg
panel2_NVR_ch2_jpg_20230329172100.jpg
panel2_NVR_ch2_jpg_20230329172800.jpg
panel2_NVR_ch2_jpg_20230403135000.jpg
panel2_NVR_ch2_jpg_20230403164700.jpg
panel2_NVR_ch2_jpg_20230403165100.jpg
panel2_NVR_ch2_jpg_20230403165500.jpg
panel2_NVR_ch2_jpg_20230403165900.jpg
panel2_NVR_ch2_jpg_20230403170300.jpg
panel2_NVR_ch2_jpg_20230403170700.jpg
panel2_NVR_ch2_jpg_20230403171200.jpg
panel2_NVR_ch2_jpg_20230403171800.jpg
panel2_NVR_ch2_jpg_20230403172300.jpg
panel2_NVR_ch2_jpg_20230403172700.jpg
panel2_NVR_ch2_jpg_20230403173100.jpg
panel2_NVR_ch2_jpg_20230417173300.jpg
panel2_NVR_ch2_jpg_20230417173400.jpg
panel2_NVR_ch2_jpg_20230417173500.jpg
panel3_NVR_ch2_jpg_20230322164400.jpg
panel3_NVR_c

#### cloud



In [30]:
os.listdir(dataset_folders["cloud"]["seg"])

['panel1_NVR_ch1_jpg_20230322164400.jpg_62.9308.jpg',
 'panel1_NVR_ch1_jpg_20230322164800.jpg_79.6429.jpg',
 'panel1_NVR_ch1_jpg_20230322165200.jpg_95.1866.jpg',
 'panel1_NVR_ch1_jpg_20230322165600.jpg_95.0747.jpg',
 'panel1_NVR_ch1_jpg_20230322170200.jpg_92.5118.jpg',
 'panel1_NVR_ch1_jpg_20230329091700.jpg_13.1237.jpg',
 'panel1_NVR_ch1_jpg_20230329092100.jpg_9.8265.jpg',
 'panel1_NVR_ch1_jpg_20230329092500.jpg_11.9565.jpg',
 'panel1_NVR_ch1_jpg_20230329092900.jpg_7.5293.jpg',
 'panel1_NVR_ch1_jpg_20230329093300.jpg_10.9067.jpg',
 'panel1_NVR_ch1_jpg_20230329093700.jpg_16.3292.jpg',
 'panel1_NVR_ch1_jpg_20230329094200.jpg_15.4985.jpg',
 'panel1_NVR_ch1_jpg_20230329141400.jpg_15.8567.jpg',
 'panel1_NVR_ch1_jpg_20230329141800.jpg_15.2128.jpg',
 'panel1_NVR_ch1_jpg_20230329142200.jpg_16.9754.jpg',
 'panel1_NVR_ch1_jpg_20230329142600.jpg_18.0986.jpg',
 'panel1_NVR_ch1_jpg_20230329143000.jpg_17.912.jpg',
 'panel1_NVR_ch1_jpg_20230329143400.jpg_19.4052.jpg',
 'panel1_NVR_ch1_jpg_2023032914

In [31]:
os.listdir(dataset_folders["cloud"]["seg"])[-2][19:33] 

'20230524150800'

In [32]:
os.listdir(dataset_folders["cloud"]["seg"])[-1][-3:] 

'csv'

In [33]:
for f in os.listdir(dataset_folders["cloud"]["seg"]):
    if f[-3:] == "csv":
        continue 
        
    date = datetime.datetime.strptime(f[19:33],'%Y%m%d%H%M%S' ) 
     
    if len(df.loc[(df["일시"]==date)]) != 0 :
        shutil.copy2(dataset_folders["cloud"]["seg"]+f,new_dataset_folders["cloud"]["seg"]+f )  
    else : 
        print(f)

panel1_NVR_ch1_jpg_20230329172100.jpg_64.941.jpg
panel1_NVR_ch1_jpg_20230329172800.jpg_68.6664.jpg
panel1_NVR_ch1_jpg_20230417135400.jpg_24.2556.jpg
panel1_NVR_ch1_jpg_20230417135500.jpg_25.5012.jpg
panel1_NVR_ch1_jpg_20230417135600.jpg_24.9529.jpg
panel1_NVR_ch1_jpg_20230417172500.jpg_77.8961.jpg
panel1_NVR_ch1_jpg_20230424143300.jpg_96.0749.jpg


In [34]:
df_cloud = pd.read_csv(f'{dataset_folders["cloud"]["seg"]}segScore.csv')
df_cloud  

,Unnamed: 0,name,score
0,0,panel1_NVR_ch1_jpg_20230322164400.jpg,62.9308
1,1,panel1_NVR_ch1_jpg_20230322164800.jpg,79.6429
2,2,panel1_NVR_ch1_jpg_20230322165200.jpg,95.1866
3,3,panel1_NVR_ch1_jpg_20230322165600.jpg,95.0747
4,4,panel1_NVR_ch1_jpg_20230322170200.jpg,92.5118
5,5,panel1_NVR_ch1_jpg_20230329091700.jpg,13.1237
6,6,panel1_NVR_ch1_jpg_20230329092100.jpg,9.8265
7,7,panel1_NVR_ch1_jpg_20230329092500.jpg,11.9565
8,8,panel1_NVR_ch1_jpg_20230329092900.jpg,7.5293
9,9,panel1_NVR_ch1_jpg_20230329093300.jpg,10.9067


In [35]:
datetime.datetime.strptime(df_cloud.loc[0]["name"][19:33],'%Y%m%d%H%M%S' ) 

datetime.datetime(2023, 3, 22, 16, 44)

In [36]:
indexs = []

for i in df_cloud.index :
    date = datetime.datetime.strptime(df_cloud.loc[i]["name"][19:33],'%Y%m%d%H%M%S') 

    if len(df.loc[(df["일시"]==date)]) == 0 :
        print(i, panel, date)
        indexs.append(i)
        
df_cloud.drop(index=indexs, inplace=True)

34 4 2023-03-29 17:21:00
35 4 2023-03-29 17:28:00
218 4 2023-04-17 13:54:00
219 4 2023-04-17 13:55:00
220 4 2023-04-17 13:56:00
260 4 2023-04-17 17:25:00
364 4 2023-04-24 14:33:00


In [37]:
df_cloud = df_cloud.reset_index(drop =True)
df_cloud.to_csv(f'{new_dataset_folders["cloud"]["seg"]}segScore.csv') 

---

In [38]:
os.listdir(dataset_folders["cloud"]["origin"])

['panel1_NVR_ch1_jpg_20230322164400.jpg',
 'panel1_NVR_ch1_jpg_20230322164800.jpg',
 'panel1_NVR_ch1_jpg_20230322165200.jpg',
 'panel1_NVR_ch1_jpg_20230322165600.jpg',
 'panel1_NVR_ch1_jpg_20230322170200.jpg',
 'panel1_NVR_ch1_jpg_20230329091700.jpg',
 'panel1_NVR_ch1_jpg_20230329092100.jpg',
 'panel1_NVR_ch1_jpg_20230329092500.jpg',
 'panel1_NVR_ch1_jpg_20230329092900.jpg',
 'panel1_NVR_ch1_jpg_20230329093300.jpg',
 'panel1_NVR_ch1_jpg_20230329093700.jpg',
 'panel1_NVR_ch1_jpg_20230329094200.jpg',
 'panel1_NVR_ch1_jpg_20230329141400.jpg',
 'panel1_NVR_ch1_jpg_20230329141800.jpg',
 'panel1_NVR_ch1_jpg_20230329142200.jpg',
 'panel1_NVR_ch1_jpg_20230329142600.jpg',
 'panel1_NVR_ch1_jpg_20230329143000.jpg',
 'panel1_NVR_ch1_jpg_20230329143400.jpg',
 'panel1_NVR_ch1_jpg_20230329144200.jpg',
 'panel1_NVR_ch1_jpg_20230329143800.jpg',
 'panel1_NVR_ch1_jpg_20230329144600.jpg',
 'panel1_NVR_ch1_jpg_20230329145000.jpg',
 'panel1_NVR_ch1_jpg_20230329145400.jpg',
 'panel1_NVR_ch1_jpg_2023032914580

In [39]:
os.listdir(dataset_folders["cloud"]["origin"])[-2][19:33] 

'20230524150700'

In [40]:
for f in os.listdir(dataset_folders["cloud"]["origin"]):
    if f[-3:] == "csv":
        continue 
        
    date = datetime.datetime.strptime(f[19:33],'%Y%m%d%H%M%S' ) 
     
    if len(df.loc[(df["일시"]==date)]) != 0 :
        shutil.copy2(dataset_folders["cloud"]["origin"]+f,new_dataset_folders["cloud"]["origin"]+f )  
    else :    
        print(f)

panel1_NVR_ch1_jpg_20230329172100.jpg
panel1_NVR_ch1_jpg_20230329172800.jpg
panel1_NVR_ch1_jpg_20230417135400.jpg
panel1_NVR_ch1_jpg_20230417135500.jpg
panel1_NVR_ch1_jpg_20230417135600.jpg
panel1_NVR_ch1_jpg_20230417172500.jpg
panel1_NVR_ch1_jpg_20230424143300.jpg


---

In [41]:
df_cloud = pd.read_csv(f'{dataset_folders["cloud"]["classification"]}classificationScore.csv')
df_cloud  

,Unnamed: 0,name,A-sky,B-pattern,C-thick-dark,D-thick-white,E-veil
0,0,panel1_NVR_ch1_jpg_20230322164400.jpg,4.424425e-06,1.453420e-03,12.288953,85.415357,2.294232
1,1,panel1_NVR_ch1_jpg_20230322164800.jpg,1.774776e-08,5.227813e-03,1.871882,97.819990,0.302909
2,2,panel1_NVR_ch1_jpg_20230322165200.jpg,1.740531e-09,3.397775e-02,39.807442,58.484578,1.673997
3,3,panel1_NVR_ch1_jpg_20230322165600.jpg,3.556563e-11,4.018900e-01,0.855924,98.492891,0.249288
4,4,panel1_NVR_ch1_jpg_20230322170200.jpg,2.254395e-18,3.120831e-02,52.248859,47.716528,0.003402
5,5,panel1_NVR_ch1_jpg_20230329091700.jpg,2.822049e-02,1.796948e-04,14.607355,0.442075,84.922165
6,6,panel1_NVR_ch1_jpg_20230329092100.jpg,2.369767e-03,3.498847e-05,20.090453,0.339599,79.567534
7,7,panel1_NVR_ch1_jpg_20230329092500.jpg,1.896276e-03,2.181109e-04,29.134721,0.314629,70.548540
8,8,panel1_NVR_ch1_jpg_20230329092900.jpg,6.471734e-05,1.116257e-04,49.240392,0.063785,50.695640
9,9,panel1_NVR_ch1_jpg_20230329093300.jpg,3.099385e-05,1.818397e-04,74.216068,0.032860,25.750858


In [42]:
datetime.datetime.strptime(df_cloud.loc[0]["name"][19:33],'%Y%m%d%H%M%S' ) 

datetime.datetime(2023, 3, 22, 16, 44)

In [43]:
indexs = []

for i in df_cloud.index :
    date = datetime.datetime.strptime(df_cloud.loc[i]["name"][19:33],'%Y%m%d%H%M%S') 

    if len(df.loc[(df["일시"]==date)]) == 0 :
        print(i, panel, date)
        indexs.append(i)
        
df_cloud.drop(index=indexs, inplace=True)

34 4 2023-03-29 17:21:00
35 4 2023-03-29 17:28:00
218 4 2023-04-17 13:54:00
219 4 2023-04-17 13:55:00
220 4 2023-04-17 13:56:00
260 4 2023-04-17 17:25:00
364 4 2023-04-24 14:33:00


In [44]:
df_cloud = df_cloud.reset_index(drop =True)
df_cloud.to_csv(f'{new_dataset_folders["cloud"]["classification"]}classificationScore.csv') 